# Homework 3
-   **Name:**  Victor Hugo Gomez Soto 
-  **e-mail:** -- victor.gomez2701@alumnos.udg.mx --


# MODULES

In [56]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


# Activity 1: Path length - BM1 vs BM2 vs CRW

In [57]:

# 1. Brownian Motion (BM) Trajectory with Fixed Step Size
def brownian_motion(n_steps, step_size=1):
    angles = np.random.uniform(0, 2*np.pi, n_steps)
    steps = np.column_stack((step_size * np.cos(angles), step_size * np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y'])

# 2. Correlated Random Walk (CRW) with High Correlation
def correlated_random_walk(n_steps, correlation=50):
    angles = np.cumsum(np.random.vonmises(0, correlation, n_steps))
    steps = np.column_stack((np.cos(angles), np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y'])

# 3. Path Length Calculation (Cumulative)
def cumulative_path_length(traj):
    step_lengths = np.sqrt(np.diff(traj['x'])**2 + np.diff(traj['y'])**2)
    return np.cumsum(step_lengths)

# Generate trajectories with same values as PDF reference
bm_traj1 = brownian_motion(1000, step_size=1)
bm_traj2 = brownian_motion(1000, step_size=1.2)
crw_traj = correlated_random_walk(1000, correlation=30)

# Compute cumulative path lengths
bm_length1 = cumulative_path_length(bm_traj1)
bm_length2 = cumulative_path_length(bm_traj2)
crw_length = cumulative_path_length(crw_traj)

# Time steps
time_steps = np.arange(1, len(bm_length1) + 1)

# Plot Path Length Comparison to match PDF style exactly
fig_path = go.Figure()
fig_path.add_trace(go.Scatter(x=time_steps, y=bm_length1, mode='lines', line=dict(width=2, color='blue'), name='Path Length BM 3'))
fig_path.add_trace(go.Scatter(x=time_steps, y=bm_length2, mode='lines', line=dict(width=2, color='red'), name='Path Length BM 6'))
fig_path.add_trace(go.Scatter(x=time_steps, y=crw_length, mode='lines', line=dict(width=2, color='green'), name='Path Length CRW 6'))
fig_path.update_layout(title='Cumulative Path Length Comparison', xaxis_title='Time Steps', yaxis_title='Path Length', legend=dict(x=1, y=1))
fig_path.show()





# Activity 2: Lévy Distribution - N Different Curves

In [58]:


# 1. Brownian Motion (BM) Trajectory with Fixed Step Size
def brownian_motion(n_steps, step_size=1):
    angles = np.random.uniform(0, 2*np.pi, n_steps)
    steps = np.column_stack((step_size * np.cos(angles), step_size * np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y'])

# 2. Correlated Random Walk (CRW) with Stronger Correlation
def correlated_random_walk(n_steps, correlation=0.999, step_size=5):
    angles = np.cumsum(np.random.vonmises(0, correlation, n_steps))
    steps = np.column_stack((step_size * np.cos(angles), step_size * np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y'])

# 3. Mean Squared Displacement (MSD) Calculation
def mean_squared_displacement(traj):
    msd = np.array([(traj.iloc[i]['x'] - traj.iloc[0]['x'])**2 + (traj.iloc[i]['y'] - traj.iloc[0]['y'])**2 for i in range(len(traj))])
    return msd

# Generate trajectories with adjusted values
bm_traj = brownian_motion(1000, step_size=2)  # BM 6
crw_traj = correlated_random_walk(1000, correlation=0.999, step_size=5)  # CRW 6 with stronger correlation

# Compute MSD
bm_msd = mean_squared_displacement(bm_traj)
crw_msd = mean_squared_displacement(crw_traj)

# Apply a moving average to smooth the CRW MSD
def moving_average(data, window_size=20):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

crw_msd_smoothed = moving_average(crw_msd, window_size=50)
bm_msd_smoothed = moving_average(bm_msd, window_size=50)

# Time steps (adjusted for moving average window)
time_steps = np.arange(len(crw_msd_smoothed))

# Plot MSD Comparison to match PDF style
fig_msd = go.Figure()
fig_msd.add_trace(go.Scatter(x=time_steps, y=bm_msd_smoothed, mode='lines', line=dict(width=2, color='blue'), name='MSD BM 6'))
fig_msd.add_trace(go.Scatter(x=time_steps, y=crw_msd_smoothed, mode='lines', line=dict(width=2, color='red'), name='MSD CRW 6 c=0.9'))
fig_msd.update_layout(title='Mean Squared Displacement Comparison', xaxis_title='Time Steps', yaxis_title='MSD', legend=dict(x=1, y=1))
fig_msd.show()



# Activity 3: Histograms + Curves


In [65]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import cauchy

# 1. Correlated Random Walk (CRW) with Cauchy-distributed Turning Angles
def correlated_random_walk_cauchy(n_steps, cauchy_scale, step_size=1):
    angles = np.cumsum(cauchy.rvs(scale=cauchy_scale, size=n_steps))
    steps = np.column_stack((step_size * np.cos(angles), step_size * np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y']), angles

# 2. Compute Turning Angles
def compute_turning_angles(angles):
    turning_angles = np.diff(angles)
    turning_angles = np.mod(turning_angles + np.pi, 2*np.pi) - np.pi  # Normalize to [-π, π]
    return turning_angles

# 3. Generate CRW Trajectories with Different Cauchy Coefficients
crw_traj_06, angles_06 = correlated_random_walk_cauchy(1000, cauchy_scale=0.6, step_size=2)
crw_traj_09, angles_09 = correlated_random_walk_cauchy(1000, cauchy_scale=0.9, step_size=2)

# 4. Compute Observed Turning Angles
turning_angles_06 = compute_turning_angles(angles_06)
turning_angles_09 = compute_turning_angles(angles_09)

# 5. Generate Theoretical Cauchy Distributions
x_vals = np.linspace(-np.pi, np.pi, 1000)
cauchy_pdf_06 = cauchy.pdf(x_vals, scale=0.6)
cauchy_pdf_09 = cauchy.pdf(x_vals, scale=0.9)

# 6. Plot Turning-angle Distribution (Observed vs Theoretical)
fig_turning = go.Figure()
fig_turning.add_trace(go.Histogram(x=turning_angles_06, histnorm='probability density', nbinsx=50, name='Observed 0.6', opacity=0.5, marker_color='blue'))
fig_turning.add_trace(go.Histogram(x=turning_angles_09, histnorm='probability density', nbinsx=50, name='Observed 0.9', opacity=0.5, marker_color='red'))
fig_turning.add_trace(go.Scatter(x=x_vals, y=cauchy_pdf_06, mode='lines', line=dict(width=2, color='blue'), name='Cauchy 0.6'))
fig_turning.add_trace(go.Scatter(x=x_vals, y=cauchy_pdf_09, mode='lines', line=dict(width=2, color='red'), name='Cauchy 0.9'))
fig_turning.update_layout(title='Turning-angle Distribution (Observed vs Theoretical)', xaxis_title='Turning Angle (radians)', yaxis_title='Probability Density', barmode='overlay')
fig_turning.show()



# Activity 4: Lévy Flight - Vec2d - 1 Trajectory

In [60]:
def levy_flight(n_steps=1000, alpha=1.5, scale=1.0, c = 0.5):
    pos = Vec2d(0, 0)
    trajectory = [pos.to_tuple()]
    angle = 0  # Ángulo inicial en radianes
    for i in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))  # Tamaño del paso con Lévy
        delta_angle = wrapcauchy.rvs(c, scale=scale)  # Generar un ángulo con distribución de Cauchy
        angle += delta_angle
        step = Vec2d(step_size, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())

    print("Primeros 5 puntos de la trayectoria:", trajectory[:5])  # Verifica si hay datos

    x, y = zip(*trajectory)
    z = np.linspace(0, 1, len(x))  # Crear un eje Z para la visualización 3D
    
    print("Cantidad de puntos generados:", len(x))  # Debe ser n_steps + 1

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', name='Lévy Flight'))
    fig.show()

# Llamada de prueba
levy_flight()


NameError: name 'Vec2d' is not defined

# Activity 5: Correlated Random Walk - Vec2d - N Trajectories

In [ ]:
def multiple_crw(n_traj=5, n_steps=1000, scale=1.0 , c = 0.5):
    fig = go.Figure()


    for i in range(n_traj):
        pos = Vec2d(0, 0)
        trajectory = [pos.to_tuple()]
        angle = 0
        
        for _ in range(n_steps):
            delta_angle = wrapcauchy.rvs(c, scale=scale * (i + 1))  # Diferente coeficiente de Cauchy para cada trayectoria
            angle += delta_angle
            step = Vec2d(1, 0).rotated(angle)
            pos += step
            trajectory.append(pos.to_tuple())

        x, y = zip(*trajectory)
        z = np.full(len(x), i)  # Usamos un valor fijo de Z para separar las trayectorias en 3D

        fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', name=f'CRW {i+1}'))

    print(f"Generadas {n_traj} trayectorias con {n_steps} pasos cada una.")
    
    fig.update_layout(title="Multiple Correlated Random Walks",
                      scene=dict(zaxis_title="Trajectory ID"),
                      showlegend=True)
    fig.show()

# Llamada de prueba
multiple_crw()


Generadas 5 trayectorias con 1000 pasos cada una.


# Correlated Random Walks and Lévy Flights

## **Exercise 1: Correlated Random Walk (CRW)**
### **What is it?**  
A **Correlated Random Walk** (CRW) is a type of random walk where each step is influenced by the previous one. Unlike a purely random movement, in CRW the turns have some **correlation**, meaning the trajectory is not completely erratic.

### **How does it work?**
1. **Trajectory generation:**  
   - We start at position `(0,0)`.
   - A list is created to store trajectory points.

2. **Movement generation:**  
   - At each step, a **random angle** is drawn from the **Cauchy distribution** (this distribution allows occasional large turns).
   - A **movement vector** is calculated based on this angle.
   - The new vector is added to the previous position.

3. **Visualization:**  
   - The generated points are stored in `x` and `y` lists.
   - The trajectory is plotted using `plotly` in 2D.

### **What is it used for?**
- **Animal and robotic search models:**  
  - Many animals (like ants and seabirds) follow similar strategies when searching for food.
  - Autonomous robots can use CRW to efficiently explore areas.
  
- **Disease propagation models:**  
  - CRW is used to simulate how diseases spread among moving populations.

- **Human mobility studies:**  
  - Helps model movement patterns in cities to optimize **traffic flow and transportation systems**.

### **Real-world example:**  
Imagine a drone exploring an area to detect radio signals. If it moved in completely random directions, it would take longer to cover the area efficiently. A **CRW** allows it to maintain some coherence in its trajectory, improving exploration.

---

## **Exercise 4: Lévy Flight**
### **What is it?**  
A **Lévy flight** is a special type of random walk where most steps are small, but **occasionally** a large jump occurs. This allows the trajectory to efficiently cover large areas.

### **How does it work?**
1. **Each step has a random size** drawn from the **Lévy distribution**.
2. **The step direction is random**, simulating an exploratory flight.
3. **The trajectory is plotted in 3D using `plotly`**.

### **What is it used for?**
- **Optimization of search strategies in nature:**  
  - Animals like **sharks, seabirds, and monkeys** have been observed following Lévy flight patterns when searching for food.
  - Inspired **AI search algorithms** for data exploration.

- **Financial models:**  
  - Used in economics to describe abrupt movements in stock markets (price jumps in financial data).

- **Wireless network design:**  
  - Helps optimize **sensor placement** and **signal distribution** in wireless networks.

### **Real-world example:**  
If you are looking for a restaurant in a new city, you might walk for a while in one area, but if you don’t find anything interesting, you **make a big jump** to another location. This is how many animals search for food in nature.

---

## **Exercise 5: Multiple Correlated Random Walks (N CRW Trajectories)**
### **What is it?**  
Here we generate **multiple correlated random walks**, each with a different coefficient. This allows us to **compare how different levels of randomness affect movement**.

### **How does it work?**
1. **Multiple trajectories are generated**, each with a different randomness level.
2. **Each trajectory uses a different Cauchy coefficient.**
3. **All are visualized together in a 3D plot using `plotly`.**

### **What is it used for?**
- **Comparison of search strategies:**  
  - If several people are exploring a forest, some might walk in straight paths, while others might take more turns. This experiment helps compare different search strategies.

- **Diffusion and transport models:**  
  - Used in **physics** to model how **particles or molecules** move in turbulent environments.

- **Optimization in artificial intelligence:**  
  - Some **AI algorithms** use this type of strategy to **efficiently explore** possible solutions.

### **Real-world example:**  
Imagine a group of **robots exploring Mars** for signs of water. Some will follow straight paths, while others will explore more freely. This model helps study **which strategy is more efficient**.
